In [2]:
data_path = '../data/kaggle-drdataboston/attempt_2'
fname = 'timewise_5s_500p'

X = np.load(f'{data_path}/{fname}_X.npy')  # shape: (N, T)
y = np.load(f'{data_path}/{fname}_y.npy')  # shape: (N,)

print(X.shape, y.shape)


(26088, 500) (26088,)


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle


2025-04-07 00:01:29.832947: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-07 00:01:29.833423: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-07 00:01:29.836292: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-07 00:01:29.843482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743973289.855905   47647 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743973289.85

In [3]:
import numpy as np

# Step 1: Get the first 20 unique labels
unique_labels = np.unique(y)
selected_labels = np.random.choice(unique_labels, 20, replace=False)

# Step 2: Create a mask for those labels
mask = np.isin(y, selected_labels)

# Step 3: Apply the mask to X and y
X_subset = X[mask]
y_subset = y[mask]

print(f"Subset size: {X_subset.shape[0]} samples from {len(np.unique(y_subset))} classes")


Subset size: 5828 samples from 20 classes


In [6]:
from sklearn.preprocessing import LabelEncoder
# Fit on the full set of labels (before train/test split)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Then split the encoded labels
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [7]:
scaler = StandardScaler()

# Reshape for scaler: (N, T) → (N*T,)
X_train_reshaped = X_train.reshape(X_train.shape[0], -1)
X_val_reshaped = X_val.reshape(X_val.shape[0], -1)
X_test_reshaped = X_test.reshape(X_test.shape[0], -1)

# Fit on training only
scaler.fit(X_train_reshaped)

X_train_scaled = scaler.transform(X_train_reshaped)
X_val_scaled = scaler.transform(X_val_reshaped)
X_test_scaled = scaler.transform(X_test_reshaped)


In [8]:
# Now you can get num_classes safely
num_classes = len(np.unique(y_encoded))

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization

model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.1),

    Dense(256, activation='relu'),
    Dropout(0.1),

    Dense(128, activation='relu'),

    Dense(num_classes, activation='softmax')
])


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


2025-04-07 00:01:40.011390: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [10]:
X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_val_seq   = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))
X_test_seq  = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, Input

model3 = Sequential([
    Input(shape=(X_train_seq.shape[1], 1)),  # (time_steps, 1)

    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Conv1D(128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])


In [13]:
from tensorflow.keras.callbacks import EarlyStopping

model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


model3.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=30,
    batch_size=32,
    verbose=1,
    callbacks = [early_stop]
)


Epoch 1/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 15s 25ms/step - accuracy: 0.0195 - loss: 4.4669 - val_accuracy: 0.0779 - val_loss: 4.0472
Epoch 2/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.0944 - loss: 3.9060 - val_accuracy: 0.2257 - val_loss: 3.2566
Epoch 3/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.2327 - loss: 3.1246 - val_accuracy: 0.3731 - val_loss: 2.4769
Epoch 4/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.3654 - loss: 2.4273 - val_accuracy: 0.5201 - val_loss: 1.8542
Epoch 5/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.4684 - loss: 1.9125 - val_accuracy: 0.5960 - val_loss: 1.5532
Epoch 6/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.5489 - loss: 1.6007 - val_accuracy: 0.6307 - val_loss: 1.3794
Epoch 7/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.5966 - loss: 1.4080 - val_accuracy: 0.6616 - val_loss: 1.2767
Epoch 8/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.6427 - loss: 1.2338 - 

In [81]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, Input

model4 = Sequential([
    Input(shape=(X_train_seq.shape[1], 1)),  # (time_steps, 1)

    Conv1D(64, kernel_size=8, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Conv1D(128, kernel_size=8, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])


In [82]:

model4.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


model4.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks = [early_stop]
)


Epoch 1/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 15s 25ms/step - accuracy: 0.0181 - loss: 4.4709 - val_accuracy: 0.0741 - val_loss: 4.0417
Epoch 2/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.0813 - loss: 3.9252 - val_accuracy: 0.2004 - val_loss: 3.3003
Epoch 3/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.1919 - loss: 3.2476 - val_accuracy: 0.3289 - val_loss: 2.7193
Epoch 4/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.3031 - loss: 2.6937 - val_accuracy: 0.4396 - val_loss: 2.1625
Epoch 5/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 17s 29ms/step - accuracy: 0.3857 - loss: 2.2435 - val_accuracy: 0.5188 - val_loss: 1.8308
Epoch 6/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 17s 29ms/step - accuracy: 0.4575 - loss: 1.9549 - val_accuracy: 0.5742 - val_loss: 1.6349
Epoch 7/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 17s 29ms/step - accuracy: 0.4979 - loss: 1.7825 - val_accuracy: 0.5883 - val_loss: 1.5585
Epoch 8/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 17s 29ms/step - accuracy: 0.5334 - loss: 1.6137 - 

In [68]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=32,
    callbacks = [early_stop]
)


Epoch 1/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.0436 - loss: 4.3062 - val_accuracy: 0.1114 - val_loss: 3.5311
Epoch 2/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.1266 - loss: 3.4662 - val_accuracy: 0.1684 - val_loss: 3.2115
Epoch 3/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.1783 - loss: 3.1753 - val_accuracy: 0.2027 - val_loss: 3.0370
Epoch 4/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2110 - loss: 2.9973 - val_accuracy: 0.2318 - val_loss: 2.9066
Epoch 5/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2296 - loss: 2.8706 - val_accuracy: 0.2443 - val_loss: 2.8338
Epoch 6/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2556 - loss: 2.7435 - val_accuracy: 0.2711 - val_loss: 2.7353
Epoch 7/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2790 - loss: 2.6268 - val_accuracy: 0.2875 - val_loss: 2.6642
Epoch 8/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.3040 - loss: 2.5336 - val_accu

Epoch 1/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 114s 198ms/step - accuracy: 0.0159 - loss: 4.4816 - val_accuracy: 0.0212 - val_loss: 4.3370
Epoch 2/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 118s 207ms/step - accuracy: 0.0215 - loss: 4.3737 - val_accuracy: 0.0230 - val_loss: 4.4823
Epoch 3/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 117s 204ms/step - accuracy: 0.0162 - loss: 4.4596 - val_accuracy: 0.0189 - val_loss: 4.3675
Epoch 4/100
 56/571 ━━━━━━━━━━━━━━━━━━━━ 1:38 191ms/step - accuracy: 0.0152 - loss: 4.3791

KeyboardInterrupt: 

In [62]:
model2 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation='relu'),

    Dense(num_classes, activation='softmax')
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [63]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


history = model2.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=32,
    callbacks = [early_stop]
)


Epoch 1/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.0307 - loss: 4.4812 - val_accuracy: 0.0851 - val_loss: 3.8637
Epoch 2/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0922 - loss: 3.8070 - val_accuracy: 0.1431 - val_loss: 3.4890
Epoch 3/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1301 - loss: 3.5112 - val_accuracy: 0.1574 - val_loss: 3.3113
Epoch 4/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1422 - loss: 3.3814 - val_accuracy: 0.1820 - val_loss: 3.2126
Epoch 5/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1634 - loss: 3.2640 - val_accuracy: 0.1935 - val_loss: 3.1618
Epoch 6/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1764 - loss: 3.1995 - val_accuracy: 0.2006 - val_loss: 3.1071
Epoch 7/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1843 - loss: 3.1495 - val_accuracy: 0.2113 - val_loss: 3.0608
Epoch 8/100
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1988 - loss: 3.0595 - val_accu

In [69]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")


123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4224 - loss: 2.2293
Test accuracy: 0.42


In [70]:
test_loss, test_acc = model2.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")


123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 953us/step - accuracy: 0.3257 - loss: 2.5740
Test accuracy: 0.33


In [78]:
test_loss, test_acc = model3.evaluate(X_test_seq, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")

model3.save('../data/kaggle-drdataboston/attempt_2/model3-cnn.keras')


123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7014 - loss: 1.1963
Test accuracy: 0.70


In [83]:
test_loss, test_acc = model4.evaluate(X_test_seq, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")

#model4.save('../data/kaggle-drdataboston/attempt_2/model4-cnn.keras')


123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7118 - loss: 1.1274
Test accuracy: 0.72
